# 02 · Parcele cu Fields of The World (pe A100)
Descarcă două scene Sentinel-2 (primăvară: aprilie–mai; vară: 1–18 iulie) de pe Planetary Computer, rulează modelul FTW și unește poligoanele în `data/out/parcels.parquet` (EPSG:3844).

**Cum rulează:** partea GPU pe Colab, cu `scripts/ftw_colab.py` (fără credențiale pe VM); poligoanele se aduc în `data/interim/`, iar notebook-ul rulează local și face unirea și verificarea.

**Atenție:**
- Tile-ul 34TGP acoperă singur toată zona; `select_tiles` adaugă alte tile-uri doar dacă aduc suprafață nouă.
- Rasterul se cere cu marjă peste zonă: `ftw-tools` pierde ultimul rând de ferestre din nord (~8 km).
- Fereastra de vară se oprește pe 18 iulie: după ~23.07.2024 scenele au procesarea 05.11, iar ftw-tools 1.4.3 nu combină versiuni diferite.

In [1]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul atlas-seceta-2024): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/atlas-seceta-2024.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/atlas-seceta-2024'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/atlas-seceta-2024'], check=True)
        os.chdir('/content/atlas-seceta-2024')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

Lucrez în C:\Users\lefpa\Downloads\gisday2026\gisday2026


In [2]:
# ftw-tools cere Python >=3.11,<3.13: rulează mereu dintr-un venv separat cu Python 3.12 (uv), oricare ar fi kernelul
os.environ['MPLBACKEND'] = 'Agg'   # evită eroarea matplotlib_inline în subprocesul ftw
v = sys.version_info
print(f'Python kernel {v.major}.{v.minor} (ftw rulează separat, cu Python 3.12)')
ON_COLAB = os.path.exists('/content')
FTW_ENV = '/content/ftwenv' if ON_COLAB else os.path.join(os.getcwd(), '.venv-ftw')
FTW = f'{FTW_ENV}/bin/ftw' if ON_COLAB else os.path.join(FTW_ENV, 'Scripts', 'ftw.exe')

Python kernel 3.12 (ftw rulează separat, cu Python 3.12)


In [3]:
# Mediul FTW (o dată pe runtime); FTW_PKGS din gisday_config = combinația testată pe 06.11.2026
if not os.path.exists(FTW):
    if not ON_COLAB:
        raise RuntimeError('Pe PC: uv venv --python 3.12 .venv-ftw && uv pip install --python .venv-ftw/Scripts/python.exe '
                           + ' '.join(FTW_PKGS))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--clear', '--python', '3.12', FTW_ENV], check=True)  # --clear: reia un venv rămas incomplet
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '-q', '--python', f'{FTW_ENV}/bin/python', *FTW_PKGS], check=True)
print('ftw:', FTW)

ftw: C:\Users\lefpa\Downloads\gisday2026\gisday2026\.venv-ftw\Scripts\ftw.exe


In [4]:
%pip install -q pystac-client planetary-computer geopandas pyarrow shapely earthengine-api

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Verifică opțiunile reale ale CLI-ului înainte de rulare (nu ghicim flag-urile)
for sub in (['--help'], ['inference', 'download', '--help'], ['inference', 'run', '--help'],
            ['inference', 'polygonize', '--help'], ['model', 'download', '--help']):
    r = subprocess.run([FTW, *sub], capture_output=True, text=True)
    print('$ ftw', ' '.join(sub)); print(r.stdout[-1500:] or r.stderr[-1500:]); print('-' * 60)

$ ftw --help
Usage: ftw [OPTIONS] COMMAND [ARGS]...

  Fields of The World (FTW) - Command Line Interface

Options:
  --help  Show this message and exit.

Commands:
  data       Downloading, unpacking, and preparing the FTW dataset.
  inference  Inference-related commands.
  model      Training and testing FTW models.

------------------------------------------------------------


$ ftw inference download --help
Usage: ftw inference download [OPTIONS]

  Download 2 Sentinel-2 scenes & stack them in a single file for inference.

Options:
  --win_a TEXT     URL to or Microsoft Planetary Computer ID of an Sentinel-2
                   L2A STAC item for the window A image  [required]
  --win_b TEXT     URL to or Microsoft Planetary Computer ID of an Sentinel-2
                   L2A STAC item for the window B image  [required]
  -o, --out TEXT   Filename to save results to  [required]
  -f, --overwrite  Overwrites the outputs if they exist
  --bbox TEXT      Bounding box to use for the download in the format
                   'minx,miny,maxx,maxy'
  --help           Show this message and exit.

------------------------------------------------------------


$ ftw inference run --help
Usage: ftw inference run [OPTIONS] INPUT

  Run inference on the stacked Sentinel-2 L2A satellite images specified via
  INPUT.

Options:
  -m, --model PATH         Path to the model checkpoint.  [required]
  -o, --out TEXT           Output filename for the inference imagery. Defaults
                           to the name of the input file name with
                           'inference.' prefix.
  --resize_factor INTEGER  Resize factor to use for inference.  [default: 2]
  --gpu INTEGER            GPU ID to use. If not provided, CPU will be used by
                           default.
  --patch_size INTEGER     Size of patch to use for inference. Defaults to
                           1024 unless the image is < 1024x1024px.
  --batch_size INTEGER     Batch size.  [default: 2]
  --padding INTEGER        Pixels to discard from each side of the patch.
  -f, --overwrite          Overwrite outputs if they exist.
  --mps_mode               Run inference in MPS mod

$ ftw inference polygonize --help
Usage: ftw inference polygonize [OPTIONS] INPUT

  Polygonize the output from inference for the raster image given via INPUT.
  Results are in the CRS of the given raster image.

Options:
  -o, --out TEXT     Output filename for the polygonized data. Defaults to the
                     name of the input file with parquet extension. Available
                     file extensions: .parquet (GeoParquet, fiboa-compliant),
                     .fgb (FlatGeoBuf), .gpkg (GeoPackage), .geojson / .json /
                     .ndjson (GeoJSON)
  --simplify FLOAT   Simplification factor to use when polygonizing in the
                     unit of the CRS, e.g. meters for Sentinel-2 imagery in
                     UTM. Set to 0 to disable simplification.  [default: 15]
  --min_size FLOAT   Minimum area size in square meters to include in the
                     output. Set to 0 to disable.  [default: 500]
  --max_size FLOAT   Maximum area size in square meters t

$ ftw model download --help
Usage: ftw model download [OPTIONS]

  Download model checkpoints

Options:
  --type [TWO_CLASS_CCBY|TWO_CLASS_FULL|THREE_CLASS_CCBY|THREE_CLASS_FULL]
                                  Short model name corresponding to a .ckpt
                                  file in github.  [required]
  -o, --out PATH                  File where the file will be stored to.
                                  Defaults to the original filename of the
                                  selected model.
  --help                          Show this message and exit.

------------------------------------------------------------


In [6]:
CKPT = INTERIM / '3_Class_FULL_FTW_Pretrained.ckpt'   # THREE_CLASS_FULL (ftw_cli/types.py); uz necomercial
if ON_COLAB and not CKPT.exists():
    subprocess.run([FTW, 'model', 'download', '--type', 'THREE_CLASS_FULL', '--out', str(CKPT)], check=True)
print('checkpoint:', CKPT, 'există' if CKPT.exists() else 'lipsește (pe PC poligoanele vin de pe Colab)')

checkpoint: C:\Users\lefpa\Downloads\gisday2026\gisday2026\data\interim\3_Class_FULL_FTW_Pretrained.ckpt lipsește (pe PC poligoanele vin de pe Colab)


In [7]:
import pystac_client, planetary_computer
from shapely.geometry import box
from gisday_lib import best_scenes, select_tiles, run_ftw_tile, parcel_coverage, coverage_gaps
cat = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                modifier=planetary_computer.sign_inplace)

def scenes(window, max_cloud=10):
    items = cat.search(collections=['sentinel-2-l2a'], bbox=AOI_BBOX, datetime=window,
                       query={'eo:cloud_cover': {'lt': max_cloud}}).item_collection()
    return best_scenes(items, AOI_BBOX)

A, B = scenes(FTW_WIN_A), scenes(FTW_WIN_B)
TILES = select_tiles(A, B, AOI_BBOX)
for t in TILES:
    pa, pb = (x.properties.get('s2:processing_baseline') for x in (A[t], B[t]))
    print(t, A[t].id, A[t].properties['eo:cloud_cover'], pa, '|', B[t].id, B[t].properties['eo:cloud_cover'], pb)
    assert pa == pb, f'{t}: versiuni de procesare diferite ({pa} vs {pb}) — ftw ar refuza perechea'

34TGP S2A_MSIL2A_20240412T092031_R093_T34TGP_20240412T143015 0.040969 05.10 | S2A_MSIL2A_20240711T092031_R093_T34TGP_20240711T152318 0.002097 05.10


In [8]:
# Pe fiecare tile: stiva A+B -> model pe GPU -> poligoane. Reluabil: poligoanele deja făcute nu se refac.
outputs = [run_ftw_tile(FTW, t, A[t], B[t], CKPT, INTERIM, AOI_BBOX, FTW_RUN) for t in TILES]
print(outputs)

[WindowsPath('C:/Users/lefpa/Downloads/gisday2026/gisday2026/data/interim/ftw_poly_34TGP_v2.parquet')]


In [9]:
import geopandas as gpd, pandas as pd
parts = []
for p in outputs:
    g = gpd.read_parquet(p) if p.suffix == '.parquet' else gpd.read_file(p)
    parts.append(g.to_crs(CRS)[['geometry']])
P = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=CRS)
P = P[P.geometry.is_valid & ~P.geometry.is_empty]
# dubluri la suprapunerea tile-urilor: păstrează poligonul mai mare când suprapunerea depășește 50% din cel mai mic
# (raportarea la cel mai mic prinde și dublurile tăiate la marginea unui tile, pe care IoU le-ar scăpa)
P['a'] = P.area
drop = set()
for i, j in zip(*P.sindex.query(P.geometry, predicate='intersects')):
    if i < j and i not in drop and j not in drop:
        gi, gj = P.geometry.iloc[i], P.geometry.iloc[j]
        if gi.intersection(gj).area / min(gi.area, gj.area) > 0.5:
            drop.add(j if P.a.iloc[i] >= P.a.iloc[j] else i)
P = P.drop(P.index[list(drop)])

In [10]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')   # afișează un link; lipești codul primit
    ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
print('Earth Engine OK')

Earth Engine OK


In [11]:
from shapely.geometry import shape as _shape
aoi_poly = gpd.GeoSeries([_shape(AOI.getInfo())], crs='EPSG:4326').to_crs(CRS).iloc[0]
P = P[P.intersects(aoi_poly)].copy()
P['geometry'] = P.intersection(aoi_poly)
P['area_ha'] = P.area / 1e4
P = P[P.area_ha >= 0.5].reset_index(drop=True)
P['parcel_id'] = range(1, len(P) + 1)
P[['parcel_id', 'area_ha', 'geometry']].to_parquet(OUT / 'parcels.parquet')
print(len(P), 'parcele;', round(P.area_ha.sum()), 'ha')

17466 parcele; 82651 ha


In [12]:
# Verificare: extinderea parcelelor și acoperirea reală a arabilului pe celule de 0,05°
import numpy as np
ext = P.to_crs(4326).total_bounds
print(f'{len(P)} parcele | {P.area_ha.sum() / 100:.0f} km² | extindere WGS84:', [round(float(v), 3) for v in ext])
assert ext[3] >= 43.99, f'Parcelele se opresc la {ext[3]:.3f}° N: banda de nord lipsește'
assert P.crs.to_epsg() == 3844 and P.parcel_id.is_unique

crop_map = ee.ImageCollection('JRC/D5/EUCROPMAP/V1').filterDate('2022-01-01', '2023-01-01').first().select(0)
arable = crop_map.gte(200).And(crop_map.lt(300)).clip(AOI).multiply(ee.Image.pixelArea()).divide(1e6).rename('km2')
xs, ys = np.round(np.arange(AOI_BBOX[0], AOI_BBOX[2] - 1e-6, 0.05), 2), np.round(np.arange(AOI_BBOX[1], AOI_BBOX[3] - 1e-6, 0.05), 2)
fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle([float(x), float(y), float(x) + 0.05, float(y) + 0.05]), {'x': float(x), 'y': float(y)})
                           for x in xs for y in ys])
feats = arable.reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=100).getInfo()['features']
T = pd.DataFrame([f['properties'] for f in feats]).rename(columns={'sum': 'arabil_km2'})
cells = gpd.GeoDataFrame(T, geometry=[box(r.x, r.y, r.x + 0.05, r.y + 0.05) for r in T.itertuples()], crs=4326).to_crs(CRS)
T['parcele_km2'] = parcel_coverage(P, cells).to_numpy()
T['acoperire'] = (T.parcele_km2 / T.arabil_km2.where(T.arabil_km2 >= 1)).round(2)
print(T.pivot(index='y', columns='x', values='acoperire').sort_index(ascending=False).to_string())
print(f'arabil EUCROPMAP: {T.arabil_km2.sum():.0f} km² | parcele: {T.parcele_km2.sum():.0f} km² ({T.parcele_km2.sum() / T.arabil_km2.sum():.0%})')
gaps = coverage_gaps(T)
assert gaps.empty, f'Celule cu arabil și fără parcele:\n{gaps[["x", "y", "arabil_km2", "parcele_km2"]]}'
print('Fără goluri: nicio celulă cu cel puțin 8 km² de arabil nu are sub 20% acoperire')

17466 parcele | 827 km² | extindere WGS84: [23.85, 43.701, 24.55, 44.001]


x      23.85  23.90  23.95  24.00  24.05  24.10  24.15  24.20  24.25  24.30  24.35  24.40  24.45  24.50
y                                                                                                      
43.95   0.70   1.46   0.59   0.31   0.36   0.22   0.33   0.57   0.53   0.55   0.81   0.60   0.75   0.91
43.90   0.87   1.24   1.51   0.53   0.59   0.54   0.45   0.56   0.57   0.80   0.66   0.52   0.79   0.86
43.85   1.10   1.69   1.91   1.40   1.12   1.12   1.06   0.89   0.61   0.39   0.59   0.55   0.54   0.90
43.80   1.23   1.04   1.81   1.22   1.05   0.68   0.72   0.67   0.88   0.57   0.57   0.69   0.80   0.84
43.75   0.90   1.00   0.75   0.36   0.60   0.77   0.52   0.69   0.57   0.71   0.71   0.84   0.75   0.76
43.70    NaN    NaN    NaN   1.07   1.44   1.11   1.37   1.88   0.97   1.04   1.04   1.90    NaN    NaN
arabil EUCROPMAP: 1088 km² | parcele: 825 km² (76%)
Fără goluri: nicio celulă cu cel puțin 8 km² de arabil nu are sub 20% acoperire


In [13]:
if ON_COLAB:     # pe PC comiți normal, din git
    push([OUT / 'parcels.parquet'], 'parcele FTW')